In [1]:
from pathlib import Path
import sys

print("Working directory:", Path.cwd())
print("Python interpreter:", sys.executable)

Working directory: <repo>\1_basic_tracing
Python interpreter: <repo>\.venv\Scripts\python.exe


# Project 1 — Basic tracing

Imagine an app that handles a request like this: **user input → prompt → LLM → parser → answer**. The app may handle many requests. Your notes call the whole app a *project*; this LangSmith workspace groups its traces under an *agent* and *environment*.

- **Project = This is our entire project** that the user interacts with. Eg : a chat bot.
- **Trace = one complete request.** When a user asks something to this chatbot, one complete trip from input to answer is a trace. It shows what happened during that execution.
- **Run = Every step that is executed in a trace.** The prompt, LLM, and parser can each appear as a run when traced. Each run can show its own input, output, duration, and error. If a request gets slow, inspect its runs to find the slow step.
- **Thread = related requests in one conversation.** A first request and a follow-up are two separate traces. Linking them with the same `thread_id` makes them one thread. Merely running the app twice does not create a thread.

```text
Thread: one conversation (only if traces are linked)
├── Trace 1: first request
│   ├── Run: format prompt
│   ├── Run: call LLM
│   └── Run: parse answer
└── Trace 2: follow-up request
    └── Runs for its recorded steps
```

**Here:** `model.invoke(prompt)` makes one request with one model run. We will inspect its trace. We have not linked multiple requests, so there is no thread yet.

In [2]:
from models.llm import get_model

prompt = Path("prompts/greeting.txt").read_text(encoding="utf-8").strip()
model = get_model()

print("Prompt:", prompt)
print("Model:", type(model).__name__)

Prompt: Write one friendly sentence welcoming a fictional new teammate named Maya. Do not invent details about her or the team.
Model: ChatGroq


In [3]:
response = model.invoke(prompt)
print(response.content)

<repo>\.venv\Lib\site-packages\langsmith\_internal\_addressing.py:104: LangSmithBetaWarning: Addressing runs with `langsmith.address` is in beta and is enabled per workspace. A workspace without it rejects the run, so the trace is lost rather than falling back to a project. The behavior may change without notice.
  _warn_once(


We're thrilled to have you join us, Maya!


### What I learned from this trace

- Input: the prompt sent to the model.
- Output: the greeting returned by the model.
- Duration: how long this call took.
- Tokens and cost: record them if LangSmith shows them.

### Trace the complete greeting request

Earlier, `model.invoke(prompt)` traced the model call by itself. 
We added `greeting_request` in `tools/request.py` so one request now shows **two recorded steps**:

1. **`greeting_request` (parent run):** Checks that the prompt is not blank, then calls the model. `@traceable` records this function.
2. **Model call (child run):** Sends the prompt to Groq and returns the greeting. LangChain records this call inside the parent run.

The next cell imports the helper, passes it the same prompt, and prints the answer. In LangSmith, this new request should appear as **one trace containing both runs**. It is separate from our earlier direct model-call trace.

In [4]:
from tools.request import greeting_request

greeting = greeting_request(prompt)
print(greeting)

Welcome to the team, Maya!


In [ ]:
greeting_request(
    "   "
)  # This is intentional. You will see a failed trace on langsmith Dashboard.

### Compare the two traces

- **Valid prompt:** [paste trace link] — `greeting_request` completed, with a model run inside it.
- **Blank prompt:** [paste trace link] — `greeting_request` failed with `Prompt must not be empty`; no model run occurred.

The blank input stopped at validation, before the request reached Groq.